# Deliverable A — Calibrated Default Probability (CatBoost 10-Fold)

**Goal:** For every applicant in `validation.csv` and `test.csv` (13,306 total), output:
- `decision` (0=decline, 1=approve)  
- `predicted_pd` — calibrated probability of default  
- `pd_lower_90`, `pd_upper_90` — 90 % conformal prediction interval  

**Decision rule:** Approve when `predicted_pd < break_even_pd`, where  
`break_even_pd = 0.0875 / (LGD + 0.0875)` (portfolio profit-maximising threshold, _not_ 0.5).

## Known traps handled
| Trap | Mitigation |
|------|------------|
| Outcome leakage | `OUTCOME_COLS` are dropped before any fit |
| `prior_decision` collider | Excluded from feature set |
| MNAR bank-feed nulls | Missingness-indicator features added |
| Reject inference | Model trained only on approved+matured; acknowledged bias |
| Calibration | Isotonic regression on held-out val set |
| Coverage | Conformal prediction (split conformal on val) |

In [1]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, 'code')   # make code/ importable from repo root

import numpy as np
import pandas as pd

# ── Module imports (all logic lives in code/) ─────────────────────────────────
from config      import SEED, DATA_DIR, SUB_DIR, N_FOLDS
from data        import load_raw, prepare_splits, engineer_features, build_feature_cols, build_matrices
from integrity   import check_integrity, check_split_leakage
from model       import run_cv
from calibration import fit_calibrator, apply_calibration, conformal_intervals, build_intervals
from decision    import compute_breakeven_pd, make_decisions
from submission  import build_submission_a

# sklearn metrics used inline in this notebook for display
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss

np.random.seed(SEED)
print('Imports OK')

Imports OK


## 1. Load data

In [2]:
train_raw, val_raw, test = load_raw()

# train: drop missing default_flag (declined/immature) upfront
# val_all: all rows → submission | val_labeled: known outcomes → calibration & optimisation
train, val_all, val_labeled, val_labeled_positions = prepare_splits(train_raw, val_raw)

print(f'train (labeled) : {len(train):>6,}  default rate {train["default_flag"].mean():.4f}')
print(f'val_all         : {len(val_all):>6,}  → submission')
print(f'val_labeled     : {len(val_labeled):>6,}  → calibration / optimisation')
print(f'test            : {len(test):>6,}')

train (labeled) : 51,722  default rate 0.1745
val_all         :  4,489  → submission
val_labeled     :  2,551  → calibration / optimisation
test            :  8,817


## 2. Integrity checks (known planted violations)

In [3]:
check_integrity(train, 'train (labeled)')
print()
check_split_leakage(train, val_all, test)

--- Integrity: train (labeled) ---
  prior_loans_default_count > prior_loans_count: 0
  days_to_default out of [1, 90]: 0
  default_flag / repayment_status mismatch: 0

--- Split leakage (business_id) ---
  train ∩ val  : 0  (expected 0)
  train ∩ test : 0  (expected 0)
  val   ∩ test : 0  (expected 0)


## 3. Feature engineering

In [4]:
train_fe       = engineer_features(train)
val_all_fe     = engineer_features(val_all)
val_labeled_fe = engineer_features(val_labeled)
test_fe        = engineer_features(test)

feature_cols, cat_indices = build_feature_cols(train_fe)
print(f'{len(feature_cols)} features,  {len(cat_indices)} categorical: {[feature_cols[i] for i in cat_indices]}')

47 features,  6 categorical: ['sector', 'geography_region', 'employee_count_bucket', 'intended_use_of_funds', 'owner_personal_credit_band', 'application_channel']


# 4. Prepare feature matrices

`train` was already filtered to labeled-only rows at load time.  \n`val_labeled_fe` is used for calibration and optimisation; `val_all_fe` feeds the submission.

**Reject inference note:** training only on approved+matured loans introduces selection bias —
The model may under-estimate PD for applicants the prior lender would have declined.
This is partially mitigated by including `prior_underwriter_score` and `has_prior_approval`\nas features, and by calibrating on the independent val set.

In [5]:
arrs = build_matrices(train_fe, val_all_fe, val_labeled_fe, test_fe, feature_cols)

print(f'X_train       : {arrs["X_train"].shape}')
print(f'X_val_labeled : {arrs["X_val_labeled"].shape}  (y_val_labeled default rate {arrs["y_val_labeled"].mean():.4f})')
print(f'X_val_all     : {arrs["X_val_all"].shape}')
print(f'X_test        : {arrs["X_test"].shape}')

X_train       : (51722, 47)
X_val_labeled : (2551, 47)  (y_val_labeled default rate 0.2062)
X_val_all     : (4489, 47)
X_test        : (8817, 47)


## 5. 10-fold stratified CatBoost cross-validation

In [6]:
oof_preds, val_preds_folds, test_preds_folds, models = run_cv(
    arrs['X_train'], arrs['y_train'],
    arrs['X_val_all'], arrs['X_test'],
    cat_indices,
)

  Fold  1/10  AUC=0.7734  trees=216
  Fold  2/10  AUC=0.7728  trees=125
  Fold  3/10  AUC=0.7719  trees=114
  Fold  4/10  AUC=0.7728  trees=186
  Fold  5/10  AUC=0.7869  trees=126
  Fold  6/10  AUC=0.7685  trees=212
  Fold  7/10  AUC=0.7835  trees=268
  Fold  8/10  AUC=0.7741  trees=201
  Fold  9/10  AUC=0.7731  trees=142
  Fold 10/10  AUC=0.7822  trees=164

  OOF AUC  : 0.7758
  Fold mean: 0.7759 ± 0.0057


## 6. Ensemble predictions + calibration

Average the 10 fold models, then calibrate with isotonic regression fitted on the 
labeled validation set (independent from training → no data leakage).

In [7]:
val_raw_all = val_preds_folds.mean(axis=1)
test_raw    = test_preds_folds.mean(axis=1)

iso = fit_calibrator(val_raw_all, arrs['y_val_labeled'], val_labeled_positions)
val_cal_all, val_cal_labeled, test_cal = apply_calibration(
    iso, val_raw_all, test_raw, val_labeled_positions
)

y_vl = arrs['y_val_labeled']
print(f'AUC     : {roc_auc_score(y_vl, val_cal_labeled):.4f}')
print(f'Logloss : {log_loss(y_vl, val_cal_labeled):.4f}')
print(f'Brier   : {brier_score_loss(y_vl, val_cal_labeled):.4f}')
print(f'Mean PD : {val_cal_labeled.mean():.4f}  (actual {y_vl.mean():.4f})')

AUC     : 0.7632
Logloss : 0.4202
Brier   : 0.1321
Mean PD : 0.2062  (actual 0.2062)


## 7. Conformal prediction intervals (90 % coverage)

**Split conformal approach** (Papadopoulos et al.):
1. Compute nonconformity scores on the labeled val set: `s_i = |y_i - p_hat_i|`
2. `q_90` = 90th percentile of {s_i}   
3. For any new point: interval = `[max(0, p - q_90), min(1, p + q_90)]`

This guarantees ≥ 90 % marginal coverage under exchangeability.

In [8]:
q_90, coverage = conformal_intervals(y_vl, val_cal_labeled)
print(f'q_90 (half-width) : {q_90:.4f}')
print(f'Empirical coverage: {coverage:.4f}  (target ≥ 0.90)')

q_90 (half-width) : 0.7375
Empirical coverage: 0.9036  (target ≥ 0.90)


## 8. Approval decision — profit-maximising threshold

In [9]:
breakeven_pd, lgd, avg_recovery = compute_breakeven_pd(train_raw)
print(f'Avg recovery  : {avg_recovery:.4f}')
print(f'LGD           : {lgd:.4f}')
print(f'Break-even PD : {breakeven_pd:.4f}  ({breakeven_pd * 100:.2f} %)')

Avg recovery  : 0.0914
LGD           : 0.9086
Break-even PD : 0.0879  (8.79 %)


## 9. Build `submission_A_decisions.csv`

In [10]:
all_pds = np.concatenate([val_cal_all, test_cal])
lower, upper = build_intervals(all_pds, q_90)
decisions    = make_decisions(all_pds, breakeven_pd)

submission, out_path = build_submission_a(
    val_all, test, val_cal_all, test_cal, decisions, lower, upper
)
print(f'Saved {len(submission):,} rows → {out_path}')
print(f'Approval rate  : {decisions.mean():.4f}  ({decisions.sum():,} approved)')
print(f'Mean PD        : {all_pds.mean():.4f}')
print(f'Mean width     : {(upper - lower).mean():.4f}')
print()
print(submission.describe())

Saved 13,306 rows → submissions/submission_A_decisions.csv
Approval rate  : 0.1656  (2,203 approved)
Mean PD        : 0.2777
Mean width     : 0.9100

           decision  predicted_pd   pd_lower_90   pd_upper_90
count  13306.000000  13306.000000  13306.000000  13306.000000
mean       0.165564      0.277664      0.010573      0.920596
std        0.371703      0.232369      0.046299      0.078156
min        0.000000      0.000000      0.000000      0.737500
25%        0.000000      0.117347      0.000000      0.854847
50%        0.000000      0.217105      0.000000      0.954605
75%        0.000000      0.365854      0.000000      1.000000
max        1.000000      1.000000      0.262500      1.000000


## 10. Validate submission format

In [11]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, 'validate_submission.py', str(SUB_DIR)],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

Expecting 13,306 applicants, 900 queries, 13x13 trajectory grid.

ISSUES
------------------------------------------------------------------------------
  [ERROR] files / missing_or_unparsable: required file 'submission_B_trajectory.csv' is missing or not valid CSV
  [ERROR] files / missing_or_unparsable: required file 'submission_C_counterfactuals.csv' is missing or not valid CSV
  [warn ] files / missing_writeup: writeup 'submission_D_writeup.pdf' not found (include it in your real submission)
------------------------------------------------------------------------------

RESULT: FAIL  (2 error(s), 1 warning(s))
Fix every ERROR above and re-run. A failing submission cannot be scored.

STDERR: 
